In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

try:
    SparkSession.builder.getOrCreate().stop()
    print("✅ Sesión antigua/rota eliminada correctamente.")
except Exception:
    print("No había sesión activa, todo limpio.")

# 1. ARRANCAR EL MOTOR (Con configuración Iceberg + MinIO)
spark = SparkSession.builder \
    .appName("Ingesta_NBA") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. LECTURA (Capa Bronze - Raw Data)
# Leemos el CSV. inferSchema=True hace que Spark adivine si es número o texto (tarda un poco más pero es cómodo)
print("Leyendo CSV masivo... espera un momento...")
ruta_csv = "/home/iceberg/notebooks/shot_logs.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(ruta_csv)

print(f"¡Leído! Tenemos {df_raw.count()} registros de tiros.")

# 3. TRANSFORMACIÓN (Capa Silver - Limpieza)
# Los nombres de columnas en Kaggle suelen ser feos. Vamos a limpiarlos y seleccionar lo útil.
# Queremos saber: Quién tiró, Distancia del tiro, Distancia del defensor, Si entró (FGM).

df_silver = df_raw.select(
    col("GAME_ID").alias("partido_id"),
    col("player_name").alias("jugador"),
    col("SHOT_DIST").alias("distancia_aro"),
    col("CLOSE_DEF_DIST").alias("distancia_defensor"),
    col("SHOT_RESULT").alias("resultado"),
    col("PTS").alias("puntos")
)

# 4. ESCRITURA EN EL LAKEHOUSE (Formato Iceberg)
# Aquí es donde ocurre la magia. Particionamos por JUGADOR.
# Esto significa que Spark organizará físicamente los archivos agrupando los de LeBron juntos.
# Esto hará que las consultas sobre jugadores específicos vuelen.

print("Guardando en Iceberg particionado por Jugador... (Esto es la Ingesta Real)")

spark.sql("CREATE DATABASE IF NOT EXISTS iceberg.nba")

df_silver.writeTo("iceberg.nba.tiros") \
    .partitionedBy(col("jugador")) \
    .createOrReplace()

print("✅ ¡Ingesta completada! Datos disponibles en el Data Lake.")

✅ Sesión antigua/rota eliminada correctamente.
Leyendo CSV masivo... espera un momento...


26/02/14 18:06:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


¡Leído! Tenemos 128069 registros de tiros.
Guardando en Iceberg particionado por Jugador... (Esto es la Ingesta Real)


26/02/14 18:06:52 ERROR Utils: Aborting task                        (0 + 1) / 1]
java.io.UncheckedIOException: Failed to close current writer
	at org.apache.iceberg.io.RollingFileWriter.closeCurrentWriter(RollingFileWriter.java:124)
	at org.apache.iceberg.io.RollingFileWriter.close(RollingFileWriter.java:147)
	at org.apache.iceberg.io.RollingDataWriter.close(RollingDataWriter.java:32)
	at org.apache.iceberg.io.FanoutWriter.closeWriters(FanoutWriter.java:82)
	at org.apache.iceberg.io.FanoutWriter.close(FanoutWriter.java:74)
	at org.apache.iceberg.io.FanoutDataWriter.close(FanoutDataWriter.java:31)
	at org.apache.iceberg.spark.source.SparkWrite$PartitionedDataWriter.close(SparkWrite.java:804)
	at org.apache.iceberg.spark.source.SparkWrite$PartitionedDataWriter.commit(SparkWrite.java:786)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$5(WriteToDataSourceV2Exec.scala:475)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:139

Py4JJavaError: An error occurred while calling o132.createOrReplace.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 7.0 failed 1 times, most recent failure: Lost task 0.0 in stage 7.0 (TID 14) (f119280d4430 executor driver): java.io.UncheckedIOException: Failed to close current writer
	at org.apache.iceberg.io.RollingFileWriter.closeCurrentWriter(RollingFileWriter.java:124)
	at org.apache.iceberg.io.RollingFileWriter.close(RollingFileWriter.java:147)
	at org.apache.iceberg.io.RollingDataWriter.close(RollingDataWriter.java:32)
	at org.apache.iceberg.io.FanoutWriter.closeWriters(FanoutWriter.java:82)
	at org.apache.iceberg.io.FanoutWriter.close(FanoutWriter.java:74)
	at org.apache.iceberg.io.FanoutDataWriter.close(FanoutDataWriter.java:31)
	at org.apache.iceberg.spark.source.SparkWrite$PartitionedDataWriter.close(SparkWrite.java:804)
	at org.apache.iceberg.spark.source.SparkWrite$PartitionedDataWriter.commit(SparkWrite.java:786)
	at org.apache.spark.

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. CONFIGURACIÓN (Conectada a tu Docker arreglado)
spark = SparkSession.builder \
    .appName("Consulta_Directa") \
    .master("local[4]") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. AUTO-REPARACIÓN SILENCIOSA (Solo si es necesario)
# Si el catálogo perdió la memoria al reiniciar Docker, esto lo regenera rápido
try:
    spark.sql("DESCRIBE iceberg.nba.tiros_test") # ¿Existe la tabla?
except Exception:
    print("⚡ Detectado catálogo vacío tras reinicio. Re-registrando metadatos...")
    spark.sql("CREATE DATABASE IF NOT EXISTS iceberg.nba")
    # Re-leemos estructura para registrarla (tarda segundos)
    if os.path.exists("/home/iceberg/notebooks/shot_logs.csv"):
        spark.read.option("header","true").option("inferSchema","true") \
             .csv("file:///home/iceberg/notebooks/shot_logs.csv") \
             .select(col("player_name").alias("jugador"), col("PTS").alias("puntos")) \
             .writeTo("iceberg.nba.tiros_test").createOrReplace()

# 3. LA CONSULTA (Lo que tú quieres ver)
print("📊 RESULTADOS:")
spark.sql("""
    SELECT 
        jugador, 
        COUNT(*) as tiros_totales,
        SUM(puntos) as puntos_totales,
        ROUND(AVG(puntos), 2) as eficiencia
    FROM iceberg.nba.tiros_test
    GROUP BY jugador
    ORDER BY puntos_totales DESC
    LIMIT 10
""").show()

print("📊 RESULTADOS 2:")
spark.sql("""
    SELECT 
        jugador, 
        COUNT(*) as tiros_totales,
        COUNT(resultado="made") as tiros_anotados,
        ROUND(AVG(puntos), 2) as eficiencia
    FROM iceberg.nba.tiros_test
    GROUP BY jugador
    ORDER BY puntos_totales DESC
    LIMIT 10
""").show()


📊 RESULTADOS:
+-----------------+-------------+--------------+----------+
|          jugador|tiros_totales|puntos_totales|eficiencia|
+-----------------+-------------+--------------+----------+
|    stephen curry|          968|          1130|      1.17|
|     james harden|         1054|          1103|      1.05|
|    klay thompson|          971|          1075|      1.11|
|     lebron james|          978|          1041|      1.06|
|       mnta ellis|         1052|          1018|      0.97|
|     kyrie irving|          942|           998|      1.06|
|   damian lillard|          986|           995|      1.01|
|lamarcus aldridge|         1050|           971|      0.92|
|   nikola vucevic|          902|           962|      1.07|
|       chris paul|          885|           947|      1.07|
+-----------------+-------------+--------------+----------+

📊 RESULTADOS 2:


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `resultado` cannot be resolved. Did you mean one of the following? [`jugador`, `puntos`].; line 5 pos 14;
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['puntos_totales DESC NULLS LAST], true
      +- 'Aggregate [jugador#210], [jugador#210, count(1) AS tiros_totales#207L, 'COUNT(('resultado = made)) AS tiros_anotados#208, round(avg(puntos#211), 2) AS eficiencia#209]
         +- SubqueryAlias iceberg.nba.tiros_test
            +- RelationV2[jugador#210, puntos#211] iceberg.nba.tiros_test iceberg.nba.tiros_test


In [ ]:
from pyspark.sql import SparkSession

# 1. Crear Sesión (Necesaria para conectar)
spark = SparkSession.builder \
    .appName("TFG_Consulta_Directa") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Tu Query Directa
print("📊 Ejecutando consulta sobre datos precargados...")

query = """
SELECT 
    jugador, 
    COUNT(*) as tiros_totales,
    SUM(CASE WHEN resultado = 'made' THEN 1 ELSE 0 END) as tiros_anotados,
    ROUND((SUM(CASE WHEN resultado = 'made' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) as porcentaje_acierto
FROM iceberg.nba.tiros_test
WHERE valor_tiro=3
GROUP BY jugador
ORDER BY porcentaje_acierto DESC
LIMIT 10
"""

spark.sql(query).show()

📊 Ejecutando consulta sobre datos precargados...
+----------------+-------------+--------------+------------------+
|         jugador|tiros_totales|tiros_anotados|porcentaje_acierto|
+----------------+-------------+--------------+------------------+
|     cody zeller|            1|             1|             100.0|
|demarcus cousins|            3|             2|             66.67|
|  nikola vucevic|            3|             2|             66.67|
|    al jefferson|            3|             2|             66.67|
|    luke babbitt|          103|            52|             50.49|
|   zach randolph|           10|             5|              50.0|
|   dwight howard|            2|             1|              50.0|
|     kyle korver|          355|           176|             49.58|
|     tony parker|           68|            32|             47.06|
|    aaron gordon|           20|             9|              45.0|
+----------------+-------------+--------------+------------------+

+-----------

26/02/06 15:46:30 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 549795 ms exceeds timeout 120000 ms
26/02/06 15:46:30 WARN SparkContext: Killing executors is not supported by current scheduler.
26/02/06 15:46:34 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [1]:
from pyspark.sql import SparkSession

# 1. Configuración de conexión (Estándar)
spark = SparkSession.builder \
    .appName("Check_Columnas") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("🔎 Inspeccionando esquema de 'iceberg.nba.tiros_test'...\n")

try:
    # Cargamos la tabla sin leer datos aún
    df = spark.table("iceberg.nba.tiros_test")

    # 1. LISTA SIMPLE
    print(f"📋 Total columnas encontradas: {len(df.columns)}")
    print(df.columns)
    
    # 2. ÁRBOL DE TIPOS (Busca aquí 'resultado')
    print("\n🌳 Estructura detallada:")
    df.printSchema()

    # 3. VERIFICACIÓN DE CONTENIDO
    print("\n👀 Muestra de la primera fila:")
    df.show(1, vertical=True, truncate=False)

except Exception as e:
    print("❌ Error: No se puede leer la tabla. ¿Se ejecutó el bootstrap?")
    print(e)

26/02/05 13:00:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


🔎 Inspeccionando esquema de 'iceberg.nba.tiros_test'...

📋 Total columnas encontradas: 20
['partido_id', 'emparejamiento', 'local_visitante', 'victoria', 'margen_final', 'numero_tiro', 'cuarto', 'tiempo_reloj', 'tiempo_posesion', 'botes', 'tiempo_con_balon', 'distancia_tiro', 'valor_tiro', 'resultado', 'defensor', 'distancia_defensor', 'tiros_metidos', 'puntos', 'jugador', 'jugador_id']

🌳 Estructura detallada:
root
 |-- partido_id: integer (nullable = true)
 |-- emparejamiento: string (nullable = true)
 |-- local_visitante: string (nullable = true)
 |-- victoria: string (nullable = true)
 |-- margen_final: integer (nullable = true)
 |-- numero_tiro: integer (nullable = true)
 |-- cuarto: integer (nullable = true)
 |-- tiempo_reloj: timestamp (nullable = true)
 |-- tiempo_posesion: double (nullable = true)
 |-- botes: integer (nullable = true)
 |-- tiempo_con_balon: double (nullable = true)
 |-- distancia_tiro: double (nullable = true)
 |-- valor_tiro: integer (nullable = true)
 |-- re

-RECORD 0--------------------------------------
 partido_id         | 21400899                 
 emparejamiento     | MAR 04, 2015 - CHA @ BKN 
 local_visitante    | A                        
 victoria           | W                        
 margen_final       | 24                       
 numero_tiro        | 1                        
 cuarto             | 1                        
 tiempo_reloj       | 2026-02-05 01:09:00      
 tiempo_posesion    | 10.8                     
 botes              | 2                        
 tiempo_con_balon   | 1.9                      
 distancia_tiro     | 7.7                      
 valor_tiro         | 2                        
 resultado          | made                     
 defensor           | Anderson, Alan           
 distancia_defensor | 1.3                      
 tiros_metidos      | 1                        
 puntos             | 2                        
 jugador            | brian roberts            
 jugador_id         | 203148            